In [0]:
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

print("Setup complete")

Setup complete


In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

dfs = []

for month in range(1, 13):
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    path = f"{RAW_PATH}/yellow_taxi/{filename}"
    
    # Read raw parquet — each file read independently (avoids schema conflict)
    df = spark.read.parquet(path)
    
    # Cast all potentially conflicting columns to consistent types
    df_cast = (df
        .withColumn("VendorID",            col("VendorID").cast("long"))
        .withColumn("passenger_count",     col("passenger_count").cast("double"))
        .withColumn("RatecodeID",          col("RatecodeID").cast("double"))
        .withColumn("PULocationID",        col("PULocationID").cast("long"))
        .withColumn("DOLocationID",        col("DOLocationID").cast("long"))
        .withColumn("payment_type",        col("payment_type").cast("long"))
        # Metadata columns
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file",         lit(filename))
        .withColumn("pipeline_name",       lit("nyc_taxi_bronze"))
    )
    
    dfs.append(df_cast)
    print(f"✅ {filename} — schema normalized")

print(f"\n✅ All 12 months ready")

✅ yellow_tripdata_2023-01.parquet — schema normalized
✅ yellow_tripdata_2023-02.parquet — schema normalized
✅ yellow_tripdata_2023-03.parquet — schema normalized
✅ yellow_tripdata_2023-04.parquet — schema normalized
✅ yellow_tripdata_2023-05.parquet — schema normalized
✅ yellow_tripdata_2023-06.parquet — schema normalized
✅ yellow_tripdata_2023-07.parquet — schema normalized
✅ yellow_tripdata_2023-08.parquet — schema normalized
✅ yellow_tripdata_2023-09.parquet — schema normalized
✅ yellow_tripdata_2023-10.parquet — schema normalized
✅ yellow_tripdata_2023-11.parquet — schema normalized
✅ yellow_tripdata_2023-12.parquet — schema normalized

✅ All 12 months ready


In [0]:
from functools import reduce
from pyspark.sql import DataFrame

df_bronze = reduce(DataFrame.union, dfs)
print(f"✅ All months unioned")
print(f"Columns: {len(df_bronze.columns)}")
df_bronze.printSchema()

✅ All months unioned
Columns: 22
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = false)

In [0]:
bronze_path = f"{RAW_PATH}/delta/bronze_yellow_taxi"

(df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("source_file")      # partition by file = easy lineage
    .save(bronze_path)
)

print("Bronze Delta table written!")

Bronze Delta table written!


In [0]:
from pyspark.sql.functions import count, when

df_verify = spark.read.format("delta").load(f"{RAW_PATH}/delta/bronze_yellow_taxi")

print(f"Total rows:    {df_verify.count():,}")
print(f"Total columns: {len(df_verify.columns)}")

# Null check
df_verify.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["VendorID", "passenger_count", "PULocationID", "DOLocationID"]
]).show()

df_verify.show(3)

Total rows:    38,310,226
Total columns: 22
+--------+---------------+------------+------------+
|VendorID|passenger_count|PULocationID|DOLocationID|
+--------+---------------+------------+------------+
|       0|        1309356|           0|           0|
+--------+---------------+------------+------------+

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------+--------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee| ingestion_timestamp|         source_file|  pipeline_name|
+--------+--------------------+-

In [0]:
print("=" * 60)
print("🥉 BRONZE LAYER RESULTS")
print("=" * 60)

df_bronze = spark.read.format("delta").load(f"{RAW_PATH}/delta/bronze_yellow_taxi")

print(f"\nTotal rows: {df_bronze.count():,}")
print(f"Total columns: {len(df_bronze.columns)}")

print("\n📋 SCHEMA:")
df_bronze.printSchema()

print("\n📊 ROW COUNT PER SOURCE FILE:")
from pyspark.sql.functions import count, col
df_bronze.groupBy("source_file") \
    .agg(count("*").alias("row_count")) \
    .orderBy("source_file") \
    .show(truncate=False)

print("\n🔍 NULL CHECK:")
from pyspark.sql.functions import count, when
df_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["VendorID", "passenger_count", "trip_distance", 
              "fare_amount", "PULocationID", "DOLocationID"]
]).show()

print("\n👀 SAMPLE ROWS:")
df_bronze.show(5, truncate=False)

🥉 BRONZE LAYER RESULTS

Total rows: 38,310,226
Total columns: 22

📋 SCHEMA:
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 